<a href="https://colab.research.google.com/github/E-tech-coder/DataScienceCapstoneProject/blob/Michi_v2/rulebasedmodel_extended_v2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Rule based model extended**

In [ ]:
!git clone https://github.com/E-tech-coder/DataScienceCapstoneProject.git

Cloning into 'DataScienceCapstoneProject'...
remote: Enumerating objects: 235, done.
remote: Counting objects: 100% (147/147), done.
remote: Compressing objects: 100% (127/127), done.
remote: Total 235 (delta 101), reused 20 (delta 20), pack-reused 88 (from 1)
Receiving objects: 100% (235/235), 1.71 MiB | 7.25 MiB/s, done.
Resolving deltas: 100% (134/134), done.


In [ ]:
%cd DataScienceCapstoneProject
!git checkout Michi_v2

/content/DataScienceCapstoneProject/DataScienceCapstoneProject/DataScienceCapstoneProject/DataScienceCapstoneProject/DataScienceCapstoneProject
Branch 'Michi_v2' set up to track remote branch 'Michi_v2' from 'origin'.
Switched to a new branch 'Michi_v2'


In [ ]:
!ls

department.csv			       linkedin-cvs-not-annotated.csv
df_profiles_cleansed.csv	       linkedin-cvs-not-annotated.json
FeatureEngineering+RandomForest.ipynb  README.md
linkedin-cvs-annotated.json	       rule_based_matching_baseline.ipynb
linkedin-cvs-annotatedV5.csv	       seniority.csv


As the "Rule-based matching (baseline)" model has several limitations, we want to improve the rule based model by adding our own defined rules.

In [ ]:
import pandas as pd
df_seniority = pd.read_csv("seniority.csv")
df_seniority.head(15)

,text,label
0,Analyst,Junior
1,Analyste financier,Junior
2,Anwendungstechnischer Mitarbeiter,Junior
3,Application Engineer,Senior
4,Applications Engineer,Senior
5,Architecte SI - Chef de projet Applicatif,Lead
6,Associate,Junior
7,Associate - Research,Junior
8,Associate Partner,Junior
9,Associate Recruiter,Junior


In [ ]:
df_department = pd.read_csv("department.csv")
df_department.head(15)

,text,label
0,Adjoint directeur communication,Marketing
1,Advisor Strategy and Projects,Project Management
2,Beratung & Projekte,Project Management
3,Beratung & Projektmanagement,Project Management
4,Beratung und Projektmanagement kommunale Partner,Project Management
5,Cadre marketing digital,Marketing
6,Chargé de communication,Marketing
7,Chargé de communication digitale,Marketing
8,Chargé de communication et marketing,Marketing
9,Chargé de Webmarketing SEO/SEA,Marketing


We use the department and seniority csv´s to define our own heuristic rules. For that we use the most frequent words per category.

In [ ]:
#but first text normalization
import re

GERMAN_MAP = str.maketrans({
    "ä": "ae", "ö": "oe", "ü": "ue", "ß": "ss",
    "Ä": "ae", "Ö": "oe", "Ü": "ue"
})

def normalize_title(x) -> str:
    if pd.isna(x):
        return ""
    s = str(x).strip()
    s = s.translate(GERMAN_MAP)
    s = s.lower()
    s = re.sub(r"\s+", " ", s)
    return s

In [ ]:
#top 20 most frequently used words for each seniority

from collections import Counter
import re
import pandas as pd
STOPWORDS = {
    "and", "of", "und", "the", "for", "in", "to", "with", "on", "at",
    "&", "/", "-", "von", "für", "des", "der", "die"
}

def tokenize(text: str) -> list:
    """Simple whitespace tokenizer."""
    return text.split()


def top_tokens_per_label(df_labels: pd.DataFrame,
                          label_col: str = "seniority",
                          text_col: str = "text",
                          top_n: int = 20) -> pd.DataFrame:

    labels = list(df_labels[label_col].dropna().unique())
    labels_sorted = sorted(labels)

    top_lists = {}
    for lab in labels_sorted:
        subset = df_labels[df_labels[label_col] == lab].copy()
        subset["title_norm"] = subset[text_col].map(normalize_title)

        counter = Counter()
        for s in subset["title_norm"]:
            counter.update(t for t in tokenize(s) if t not in STOPWORDS)

        top_lists[lab] = counter.most_common(top_n)

    #build multi-index DataFrame
    cols = []
    data = {}
    for lab in labels_sorted:
        cols.extend([(lab, "token"), (lab, "count")])
        data[(lab, "token")] = [t for t, c in top_lists[lab]]
        data[(lab, "count")] = [c for t, c in top_lists[lab]]

    result = pd.DataFrame(data)
    result.columns = pd.MultiIndex.from_tuples(cols)
    result.index = range(1, top_n + 1)
    result.index.name = "rank"
    return result

top20_seniority_table = top_tokens_per_label(df_seniority, label_col="label", text_col="text", top_n=20)
display(top20_seniority_table)


Director                           Junior        \
               token count                      token count   
rank                                                          
1           director   959                  marketing   141   
2              sales   441                     junior    86   
3          marketing   258                    analyst    67   
4           business   101                 referentin    52   
5        development    80                assistentin    52   
6           managing    70                   business    45   
7             global    70              mitarbeiterin    36   
8             senior    47                   referent    35   
9             europe    42                mitarbeiter    34   
10              emea    38                      sales    33   
11              dach    37                    manager    32   
12     international    37                   vertrieb    30   
13        management    36                        crm    28   
14           digital    35              kommunikation    18   
15           germany    29                         it    17   
16    communications    29                  associate    15   
17             group    28                  assistent    15   
18                it    26                       fuer    14   
19                 |    25  unternehmenskommunikation    12   
20        operations    22                development    12   

                   Lead                Management               Senior        
                  token count               token count          token count  
rank                                                                          
1                  head   992   geschaeftsfuehrer   143        manager  2354  
2             marketing   845               sales   124      marketing  1013  
3                leiter   762           president   120          sales   825  
4                 sales   558                vice   117         senior   557  
5              vertrieb   448           marketing   103     management   369  
6               leitung   390                 ceo    78       business   356  
7              business   184               chief    76            crm   316  
8       vertriebsleiter   184  geschaeftsfuehrung    71        account   268  
9                   crm   171             officer    68     consultant   253  
10                   it   163                  vp    67    development   245  
11             leiterin   140            business    45        project   172  
12           teamleiter   129            vertrieb    44    responsable   157  
13          development   124               owner    39             it   142  
14    geschaeftsleitung   101             founder    35            key   141  
15              digital    77          co-founder    34      assistant   136  
16        projektleiter    77         development    33        digital   132  
17        kommunikation    76             product    33           head   131  
18               global    75              global    26  communication   124  
19       bereichsleiter    74                   |    22         global   122  
20            prokurist    71           assistenz    22      managerin   113

In [ ]:
#top 20 most frequently used words for each department

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 0)

top20_department_table = top_tokens_per_label(df_department,label_col="label",text_col="text",top_n=20)
display(top20_department_table)

Administrative       Business Development            Consulting  \
                   token count                token count           token   
rank                                                                        
1            assistentin    36             business   608      consultant   
2              assistenz    31          development   295          senior   
3      geschaeftsleitung    17              manager   187             sap   
4     geschaeftsfuehrung    16                 head    77         berater   
5              assistent    10             director    59               |   
6                     gf     5                  crm    52              it   
7                 office     5               senior    50      management   
8              assistant     5              analyst    36         inhouse   
9            sekretaerin     4                   it    34     recruitment   
10                    gl     3           management    34        dynamics   
11                   ceo     3               leiter    28       microsoft   
12            management     3              digital    27         digital   
13     projektmanagement     2              process    27         trainer   
14              vorstand     2                  new    27             nav   
15                  fuer     2               global    25         manager   
16                  kfm.     2         intelligence    24           coach   
17               leitung     2              account    22  projektmanager   
18             executive     2        international    21  transformation   
19    institutsdirektion     1                 unit    20             erp   
20            vorstandes     1           consultant    19         project   

           Customer Support                   Human Resources        \
     count            token count                       token count   
rank                                                                  
1      133          support    28                          hr    18   
2       39               it    11                       human     7   
3       20          manager     7                     manager     7   
4       19         customer     6                   resources     5   
5       12        technical     5                 assistentin     3   
6       11              1st     2                    director     3   
7       10          systems     2                        head     3   
8        9     it-supporter     2                  management     2   
9        9           global     2                      office     2   
10       8             head     2                   assistant     2   
11       7          service     2                  recruiting     2   
12       7           leiter     2                   abteilung     1   
13       5          account     2  buerokauffrau/hr-assistenz     1   
14       5       management     2                          gl     1   
15       5                |     2                    heavenhr     1   
16       4             line     1                     adviser     1   
17       4          backend     1                       hr-it     1   
18       4              edv     1                   ressource     1   
19       4              erp     1               administrator     1   
20       4        supporter     1                mediaberater     1   

     Information Technology             Marketing             Other        \
                      token count           token count       token count   
rank                                                                        
1                       crm   512       marketing  3352  operations    42   
2                        it   373         manager   835     manager    12   
3                   manager   246           sales   464    director     8   
4                      head   143            head   336        head     8   
5                   digital   124  communications   290     reven

Rule-based heuristics are derived from these two lists. These rules supplement the exact string matching methodology. The most frequently occurring words are used as keywords: if such a keyword appears in the job title, it serves as an indicator for assignment to a specific seniority level or department.

#**Rules for seniority und departments**




In [ ]:
#dictionaries (Fallback)
sen_map = dict(zip(df_seniority["text"].map(normalize_title), df_seniority["label"]))
dep_map = dict(zip(df_department["text"].map(normalize_title), df_department["label"]))

print("sen_map:", len(sen_map), "entries")
print("dep_map:", len(dep_map), "entries")

sen_map: 9411 entries
dep_map: 10131 entries


In [ ]:
#check duplicates
normalized_sen_titles = df_seniority["text"].map(normalize_title)
duplicates_sen = normalized_sen_titles[normalized_sen_titles.duplicated()].unique()
if len(duplicates_sen) > 0:
    print(f"Warning: {len(duplicates_sen)} duplicate normalized seniority titles found")

normalized_dep_titles = df_department["text"].map(normalize_title)
duplicates_dep = normalized_dep_titles[normalized_dep_titles.duplicated()].unique()
if len(duplicates_dep) > 0:
    print(f"Warning: {len(duplicates_dep)} duplicate normalized department titles found")

In [ ]:
#seniority rules
def has_word(t: str, token: str) -> bool:
    return re.search(rf"\b{re.escape(token)}\b", t) is not None

def predict_seniority(title):
    t = normalize_title(title)

    #hard rules
    #management
    if any(has_word(t, k) for k in [
        "ceo","cfo","cto","cmo","chief","vice","president","owner","gesellschafter","founder","cofounder","co-founder","prokurist","prokuristin","unternehmensinhaber","gesch"]):
        return "Management", "keyword_mgmt"

    #director
    if any(has_word(t, k) for k in [
        "director", "executive director","global director","finance director","strategy director"]):
        return "Director", "keyword_director"

    #lead
    if any(has_word(t, k) for k in [
        "lead","leitung","leiter","head"]):
        return "Lead", "keyword_lead"

    #senior
    if any(has_word(t, k) for k in [
        "senior"]) or has_word(t, "sr"):
        return "Senior", "keyword_senior"

    #junior
    if any(has_word(t, k) for k in [
        "intern","junior","trainee","student","referent","auszubild"]):
        return "Junior", "keyword_junior"

    #fallback exact match in label list
    if t in sen_map:
        return sen_map[t], "list_exact"

    return None, "default"

In [ ]:
#department rules

def predict_department(title, organization=None):
    t = normalize_title(title)

    #hard rules
    #administrative
    if any(has_word(t, k) for k in [
        "assistentin","assistenz","assistent","office","assistant","sekret",
        "verwaltung", "bueroleiter", "facility", "administrador"]):
        return "Administrative", "keyword_administrative"

    #business Development
    if any(has_word(t, k) for k in [
        "business development", "business developer", "new business", "strategic partnerships"]) or has_word(t, "bd"):
        return "Business Development", "keyword_bd"

    #consulting
    if any(has_word(t, k) for k in [
        "consultant","berater","sap","dynamics","erp"]):
        return "Consulting", "keyword_consulting"

    #customer Support
    if any(has_word(t, k) for k in [
        "support","customer","supporter"]):
        return "Customer Support", "keyword_customer_support"

    #information Technology
    if any(has_word(t, k) for k in [
        "software","developer","engineer","architect","devops","cloud","data scientist","data engineer",
        "network","systems administrator","administrator"]) or has_word(t, "it"):
        return "Information Technology", "keyword_it"

    #human Resources
    if any(has_word(t, k) for k in [
        "human","resources","ressource", "hr", "personal", "recruitment", "talent", "personalleiter"]):
        return "Human Resources", "keyword_hr"

    #marketing
    if any(has_word(t, k) for k in [
        "marketing","communication","communications","kommunikation","messe","event"]):
        return "Marketing", "keyword_marketing"

    #project Management
    if any(has_word(t, k) for k in [
        "project","projektleiter","projektmanager","projektmanagement","projekt",
        "projektleitung","projects"]):
        return "Project Management", "keyword_project_mgmt"

    #purchasing
    if any(has_word(t, k) for k in [
        "einkauf","purchas","eink"]):
        return "Purchasing", "keyword_purchasing"

    #sales
    if any(has_word(t, k) for k in [
        "sales","vertrieb","vertriebsleiter","salesforce"]):
        return "Sales", "keyword_sales"

    #fallback exact match in label list
    if t in dep_map:
        return dep_map[t], "list_exact"

    return None, "default"

# **Pipeline:**

In [ ]:
def pipeline(job_title):
    dep, dep_reason = predict_department(job_title)
    sen, sen_reason = predict_seniority(job_title)
    return {
        "department": dep,
        "seniority": sen,
        "dep_reason": dep_reason,
        "sen_reason": sen_reason
    }

In [ ]:
pipeline("CMO")

{'department': None,
 'seniority': 'Management',
 'dep_reason': 'default',
 'sen_reason': 'keyword_mgmt'}

In [ ]:
tests = ["CMO", "CFO", "Senior Network Engineer", "Marketing Intern", "Human Resources Generalist", "Junior Consultant"]
for t in tests:
    print(t, "->", pipeline(t))

CMO -> {'department': None, 'seniority': 'Management', 'dep_reason': 'default', 'sen_reason': 'keyword_mgmt'}
CFO -> {'department': None, 'seniority': 'Management', 'dep_reason': 'default', 'sen_reason': 'keyword_mgmt'}
Senior Network Engineer -> {'department': 'Information Technology', 'seniority': 'Senior', 'dep_reason': 'keyword_it', 'sen_reason': 'keyword_senior'}
Marketing Intern -> {'department': 'Marketing', 'seniority': 'Junior', 'dep_reason': 'keyword_marketing', 'sen_reason': 'keyword_junior'}
Human Resources Generalist -> {'department': 'Human Resources', 'seniority': None, 'dep_reason': 'keyword_hr', 'sen_reason': 'default'}
Junior Consultant -> {'department': 'Consulting', 'seniority': 'Junior', 'dep_reason': 'keyword_consulting', 'sen_reason': 'keyword_junior'}


# **Evaluation**

In [ ]:
eval_df = pd.read_csv("df_profiles_cleansed.csv")
eval_df.head()

,organization,position,startDate,endDate,status,department,seniority,person_id,job_count,job_duration_years
0,Depot4Design GmbH,Prokurist,2019-08,2025-12,ACTIVE,Other,Management,0,6,6.339726
1,Depot4Design GmbH,CFO,2019-07,2025-12,ACTIVE,Other,Management,0,6,6.424658
2,Depot4Design GmbH,Betriebswirtin,2019-07,2025-12,ACTIVE,Other,Professional,0,6,6.424658
3,Depot4Design GmbH,Prokuristin,2019-07,2025-12,ACTIVE,Other,Management,0,6,6.424658
4,Depot4Design GmbH,CFO,2019-07,2025-12,ACTIVE,Other,Management,0,6,6.424658


In [ ]:
eval_active_df = eval_df[eval_df["status"] == "ACTIVE"].copy()

In [ ]:
#text normalization eval test set
eval_active_df["position_norm"] = eval_active_df["position"].apply(normalize_title)

In [ ]:
predictions_dep = eval_active_df["position"].apply(predict_department)
eval_active_df["dep_pred"] = predictions_dep.apply(lambda x: x[0])
eval_active_df["dep_pred_reason"] = predictions_dep.apply(lambda x: x[1])

predictions_sen = eval_active_df["position"].apply(predict_seniority)
eval_active_df["sen_pred"] = predictions_sen.apply(lambda x: x[0])
eval_active_df["sen_pred_reason"] = predictions_sen.apply(lambda x: x[1])

**Coverage Calculation**

In [ ]:
dep_coverage = (eval_active_df["dep_pred"].notna().sum() / len(eval_active_df)) * 100
sen_coverage = (eval_active_df["sen_pred"].notna().sum() / len(eval_active_df)) * 100

print(f"Department Prediction Coverage: {dep_coverage:.2f}%")
print(f"Seniority Prediction Coverage: {sen_coverage:.2f}%")

Department Prediction Coverage: 23.68%
Seniority Prediction Coverage: 44.85%


Department Prediction Coverage: Only 24% of the job titles could be classified into a department.

Seniority Prediction Coverage: 45% of the job titles could be classified into a seniority level.

This is a huge improvement in comparison with the exact string matching only, but means that for a significant portion of the job titles, the current rule-based model could still not make a prediction.

**Classification Report and Accuracy**

**metrics for only matched titles:**

In [ ]:
from sklearn.metrics import classification_report, accuracy_score

eval_dep = eval_active_df[eval_active_df["dep_pred"].notna()].copy()
eval_sen = eval_active_df[eval_active_df["sen_pred"].notna()].copy()


print(f"\n--- Department Evaluation (n={len(eval_dep)}) ---")
if len(eval_dep) > 0:
    dep_acc = accuracy_score(eval_dep["department"], eval_dep["dep_pred"])
    print(f"Accuracy: {dep_acc:.2%}")
    print("\nClassification Report:")
    print(classification_report(eval_dep["department"], eval_dep["dep_pred"], zero_division=0))

print(f"\n--- Seniority Evaluation (n={len(eval_sen)}) ---")
if len(eval_sen) > 0:
    sen_acc = accuracy_score(eval_sen["seniority"], eval_sen["sen_pred"])
    print(f"Accuracy: {sen_acc:.2%}")
    print("\nClassification Report:")
    print(classification_report(eval_sen["seniority"], eval_sen["sen_pred"], zero_division=0))


--- Department Evaluation (n=170) ---
Accuracy: 88.24%

Classification Report:
                        precision    recall  f1-score   support

        Administrative       0.62      1.00      0.77        10
  Business Development       1.00      1.00      1.00         7
            Consulting       0.96      0.89      0.92        27
      Customer Support       0.75      1.00      0.86         3
       Human Resources       0.88      1.00      0.93        14
Information Technology       0.92      0.97      0.95        36
             Marketing       0.77      0.91      0.83        11
                 Other       0.00      0.00      0.00        14
    Project Management       0.86      1.00      0.93        19
            Purchasing       1.00      1.00      1.00         2
                 Sales       0.96      0.96      0.96        27

              accuracy                           0.88       170
             macro avg       0.79      0.88      0.83       170
          weighted avg

**Department Evaluation:**

Accuracy: The model achieved an accuracy of 88% for department predictions at only matched titles (170 out of 718).

The classification report shows strong performance for categories like 'Business Development', 'Customer Support', 'Human Resources', 'Information Technology', 'Project Management', 'Purchasing', and 'Sales' with high precision and recall.

The 'other' category was never correctly predicted, because our rule based model has no rules for this category.

**Seniority Evaluation:**

Accuracy: The model achieved an accuracy of 82% for seniority predictions at only matched titles (322 out of 718).

The classification report indicates good performance for 'Director', 'Lead', and 'Management' categories.

The 'professional' category was never correctly predicted, because as already documented for the string-matching baseline, no labeled training data for this category are available.

---

**overall performance metrics (end to end):**

In [ ]:
print("\n=== Overall End-to-End Evaluation ===")

eval_overall = eval_active_df.copy()
eval_overall["dep_pred_overall"] = eval_overall["dep_pred"].fillna("No Prediction")
eval_overall["sen_pred_overall"] = eval_overall["sen_pred"].fillna("No Prediction")

# Department
print("\n--- Overall Department Evaluation ---")
y_true_dep = eval_overall["department"].fillna("Unknown_Department").astype(str)
y_pred_dep = eval_overall["dep_pred_overall"].astype(str)

dep_acc_overall = accuracy_score(y_true_dep, y_pred_dep)
print(f"Accuracy: {dep_acc_overall:.2%}")
print("\nClassification Report:")
print(classification_report(y_true_dep, y_pred_dep, zero_division=0))

# Seniority
print("\n--- Overall Seniority Evaluation ---")
y_true_sen = eval_overall["seniority"].fillna("Unknown_Seniority").astype(str)
y_pred_sen = eval_overall["sen_pred_overall"].astype(str)

sen_acc_overall = accuracy_score(y_true_sen, y_pred_sen)
print(f"Accuracy: {sen_acc_overall:.2%}")
print("\nClassification Report:")
print(classification_report(y_true_sen, y_pred_sen, zero_division=0))


=== Overall End-to-End Evaluation ===

--- Overall Department Evaluation ---
Accuracy: 20.89%

Classification Report:
                        precision    recall  f1-score   support

        Administrative       0.62      0.43      0.51        23
  Business Development       1.00      0.35      0.52        20
            Consulting       0.96      0.53      0.69        45
      Customer Support       0.75      0.43      0.55         7
       Human Resources       0.88      0.74      0.80        19
Information Technology       0.92      0.51      0.65        69
             Marketing       0.77      0.42      0.54        24
         No Prediction       0.00      0.00      0.00         0
                 Other       0.00      0.00      0.00       405
    Project Management       0.86      0.49      0.62        39
            Purchasing       1.00      0.12      0.22        16
                 Sales       0.96      0.51      0.67        51

              accuracy                         

**Overall Department Evaluation:**

The overall evaluation shows a low accuracy of 21%, which is primarily caused by the very high proportion of the Other class, which, as already mentioned, is not covered by the rule-based approach. For clearly defined departments such as Information Technology, Consulting or Human Resources, however, the model achieves high precision, but only moderate recall values. Overall, the result reflects low end-to-end coverage combined with solid rule quality for specified departments.

**Overall Seniority Evaluation:**

The overall accuracy of around 37% is significantly influenced by the Professional class, for which there are no explicit labels in the training and rule set. For the remaining seniority levels, the model achieves high precision, especially for Lead and Management, while at the same time having limited recall. The result underscores the strong dependence of end-to-end performance on label coverage and rule completeness.

---

**Extended Overall Performance Analysis (Excluding 'Other' Department and 'Professional' Seniority)**

In [ ]:
# Department (excluding 'Other')
print("\n--- Department Evaluation (Excluding 'Other') ---")
eval_dep_filtered = eval_overall[eval_overall["department"] != "Other"].copy()

if not eval_dep_filtered.empty:
    dep_acc_filtered = accuracy_score(
        eval_dep_filtered["department"],
        eval_dep_filtered["dep_pred_overall"]
    )
    print(f"Accuracy: {dep_acc_filtered:.2%}")
    print("\nClassification Report:")
    print(classification_report(
        eval_dep_filtered["department"],
        eval_dep_filtered["dep_pred_overall"],
        zero_division=0
    ))
else:
    print("No data to evaluate after excluding 'Other'.")

# Seniority (excluding 'Professional')
print("\n--- Seniority Evaluation (Excluding 'Professional') ---")
eval_sen_filtered = eval_overall[eval_overall["seniority"] != "Professional"].copy()

if not eval_sen_filtered.empty:
    sen_acc_filtered = accuracy_score(
        eval_sen_filtered["seniority"],
        eval_sen_filtered["sen_pred_overall"]
    )
    print(f"Accuracy: {sen_acc_filtered:.2%}")
    print("\nClassification Report:")
    print(classification_report(
        eval_sen_filtered["seniority"],
        eval_sen_filtered["sen_pred_overall"],
        zero_division=0
    ))
else:
    print("No data to evaluate after excluding 'Professional'.")


--- Department Evaluation (Excluding 'Other') ---
Accuracy: 47.92%

Classification Report:
                        precision    recall  f1-score   support

        Administrative       1.00      0.43      0.61        23
  Business Development       1.00      0.35      0.52        20
            Consulting       1.00      0.53      0.70        45
      Customer Support       0.75      0.43      0.55         7
       Human Resources       1.00      0.74      0.85        19
Information Technology       0.97      0.51      0.67        69
             Marketing       0.83      0.42      0.56        24
         No Prediction       0.00      0.00      0.00         0
    Project Management       0.90      0.49      0.63        39
            Purchasing       1.00      0.12      0.22        16
                 Sales       1.00      0.51      0.68        51

              accuracy                           0.48       313
             macro avg       0.86      0.41      0.54       313
          

**Extended Overall Department Evaluation (Excluding Other):**

After excluding the Other class, accuracy increases significantly to 48%, which better reflects the actual performance of rule-based department assignment. Precision is high for almost all departments, while recall remains moderate. This confirms that the rules are reliable in clear cases, but do not cover all relevant titles.

**Extended Overall Seniority Evaluation (Excluding Professional):**

Excluding the professional class increases accuracy to 59%, which highlights the quality of the seniority rules for the remaining classes. In particular, lead, management and senior are recognised with high precision and stable recall, while junior stands out due to its small sample size and weak keyword coverage. Overall, the report shows robust rule performance with sufficient label coverage.

#**Error Analysis**

In [ ]:
print("### Department Misclassifications (Excluding 'Other') ###")
department_errors = eval_active_df_overall[
    (eval_active_df_overall['department'] != eval_active_df_overall['dep_pred_overall']) &
    (eval_active_df_overall['dep_pred_overall'] != 'No Prediction') &
    (eval_active_df_overall['department'] != 'Unknown_Department') &
    (eval_active_df_overall['department'] != 'Other') #exclude 'Other' department
].copy()

#Display some common misclassifications or a sample of errors
if not department_errors.empty:
    print(f"Total Department Errors (excluding 'No Prediction', 'Unknown_Department', and 'Other' true labels): {len(department_errors)}")
    display(department_errors[['position', 'department', 'dep_pred_overall', 'dep_pred_reason']].head(10))
    print("\nTop 10 misclassified actual departments:")
    display(department_errors['department'].value_counts().head(10))
    print("\nTop 10 predicted departments in error:")
    display(department_errors['dep_pred_overall'].value_counts().head(10))
else:
    print("No department misclassifications found (or all errors are 'No Prediction' / 'Unknown_Department' / 'Other' true labels).")

### Department Misclassifications (Excluding 'Other') ###
Total Department Errors (excluding 'No Prediction', 'Unknown_Department', and 'Other' true labels): 10


,position,department,dep_pred_overall,dep_pred_reason
538,Project Lead for Swap Re-engineering Project,Project Management,Information Technology,keyword_it
608,Director of Business Analysts and Special Proj...,Information Technology,Project Management,keyword_project_mgmt
1061,Mediaberater DpS - Fachzeitschrift f. Schädlin...,Sales,Consulting,keyword_consulting
1374,Head of Regulatory Projects,Consulting,Project Management,keyword_project_mgmt
1435,Senior Enterprise IT Systems Specialist,Information Technology,Consulting,keyword_consulting
1569,Freelance Marketing Consulting / Client Servic...,Consulting,Marketing,keyword_marketing
1699,Director of Sales & Marketing,Sales,Marketing,keyword_marketing
1823,Cloud Advisory Analyst,Consulting,Information Technology,keyword_it
2078,"Utbildare och konsult inom organisation, ledar...",Consulting,Project Management,keyword_project_mgmt
2354,Executive Vice President Digital Customer Solu...,Marketing,Customer Support,keyword_customer_support



Top 10 misclassified actual departments:


,count
department,
Consulting,4
Sales,2
Information Technology,2
Project Management,1
Marketing,1



Top 10 predicted departments in error:


,count
dep_pred_overall,
Project Management,3
Information Technology,2
Consulting,2
Marketing,2
Customer Support,1


In [ ]:
print("\n### Seniority Misclassifications (Excluding 'Professional') ###")
seniority_errors = eval_active_df_overall[
    (eval_active_df_overall['seniority'] != eval_active_df_overall['sen_pred_overall']) &
    (eval_active_df_overall['sen_pred_overall'] != 'No Prediction') &
    (eval_active_df_overall['seniority'] != 'Unknown_Seniority') &
    (eval_active_df_overall['seniority'] != 'Professional') #exclude 'Professional' seniority
].copy()

#Display some common misclassifications or a sample of errors
if not seniority_errors.empty:
    print(f"Total Seniority Errors (excluding 'No Prediction', 'Unknown_Seniority', and 'Professional' true labels): {len(seniority_errors)}")
    display(seniority_errors[['position', 'seniority', 'sen_pred_overall', 'sen_pred_reason']].head(10))
    print("\nTop 10 misclassified actual seniorities:")
    display(seniority_errors['seniority'].value_counts().head(10))
    print("\nTop 10 predicted seniorities in error:")
    display(seniority_errors['sen_pred_overall'].value_counts().head(10))
else:
    print("No seniority misclassifications found (or all errors are 'No Prediction' / 'Unknown_Seniority' / 'Professional' true labels).")


### Seniority Misclassifications (Excluding 'Professional') ###
Total Seniority Errors (excluding 'No Prediction', 'Unknown_Seniority', and 'Professional' true labels): 33


,position,seniority,sen_pred_overall,sen_pred_reason
42,Managing Director,Management,Director,keyword_director
126,Managing Director,Management,Director,keyword_director
260,Managing Director,Management,Director,keyword_director
269,New Business Manager,Lead,Senior,list_exact
384,Managing Director,Management,Director,keyword_director
428,Managing Director,Management,Director,keyword_director
499,CEO Morgenpost & TAG 24 | Vertriebsleiter Säch...,Lead,Management,keyword_mgmt
654,"Executive Director, Marketing Strategy Director",Management,Director,keyword_director
960,Executive Director,Management,Director,keyword_director
965,Student Records Manager,Lead,Junior,keyword_junior



Top 10 misclassified actual seniorities:


,count
seniority,
Management,25
Lead,6
Junior,1
Senior,1



Top 10 predicted seniorities in error:


,count
sen_pred_overall,
Director,19
Senior,8
Management,3
Lead,2
Junior,1


**Key Observations:**

Rule Specificity vs. Breadth: The rules show very high precision when they make a prediction, especially for departments and the core seniority levels (Management, Lead, Director). The number of misclassifications, once 'No Prediction', 'Other', and 'Professional' are excluded, is quite low.

Rule Conflict and Prioritization: A significant portion of errors stems from conflicts between keywords. For example, 'Director' seems to be a very strong keyword, often leading to a Director prediction even when the overall title ('Managing Director') might be better classified as Management. Similarly, specific keywords (e.g., 'Student') can override broader role implications (e.g., 'Manager').

Lexical Ambiguity: Some job titles contain words that could legitimately fall into multiple categories (e.g., 'Project' in IT roles, 'Berater' in Sales roles), leading to misclassifications based on which rule is triggered first or is more strongly defined.

# **Conclusion**

The evaluation shows that the rule-based approach to seniority and department classification achieves a high degree of precision in clearly defined cases, but is limited in an end-to-end view due to its limited coverage.

Further improvements to the extended rule-based model will not be pursued at this point, as the set of rules has been largely exhausted and additional rules would primarily lead to overengineering. Rather, the results highlight the inherent limitations of rule-based methods and underscore the need to resort to more advanced, data-driven approaches for greater coverage and robustness.

*(The extended rule-based model was continuously improved iteratively over several versions. However, due to internal inconsistencies in the handling of the data sets used, not all adjustments and optimisations made could be documented in detail. The main developments and their effects are summarised in the section Model Failures.)*